# §6 — 2D RoPE in the ViT

Retrains the CLIP-on-EuroSAT setup with **2D RoPE** (each patch's `(x, y)` grid coordinates feed RoPE on `q`/`k`), then compares against the **learned PE** and **1D RoPE** runs from `rope_vs_learned.ipynb`. Includes the same 64→96 length-extrapolation test for the 2D-RoPE run.

**2D-RoPE recipe:** `head_dim` is split in half — the first half rotates by `x_coords`, the second half by `y_coords`. Inside the ViT, the CLS token gets coordinate `(0, 0)` and patches are shifted to `(px+1, py+1)` so CLS is distinguishable from the top-left patch. The per-axis cos/sin cache defaults to `4×(grid+1)` slots, which already covers 64→96 extrapolation; otherwise call `vit.set_rope_grid_size(...)`.

**Comparison:** if you already ran `rope_vs_learned.ipynb`, the `learned` and `rope` history files on Drive are reused (no retraining). Only `rope2d` is trained fresh from this notebook.

## 1. Install dependencies

In [1]:
%%capture
!pip -q install -U transformers datasets "sentence-transformers<4.0" accelerate tqdm matplotlib wandb pyyaml einops
!pip -q install --force-reinstall --no-deps --no-cache-dir pillow==11.3.0

## 2. Mount Drive and set up the repo path

In [2]:
import os, sys
from pathlib import Path

USE_DRIVE = True
DRIVE_REPO_ROOT = Path('/content/drive/MyDrive/caltech/junior/hw3/')  # edit if needed

if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=True)
    REPO_ROOT = DRIVE_REPO_ROOT
else:
    REPO_ROOT = Path('/content/hw3')

assert REPO_ROOT.exists(), f'Repo root does not exist: {REPO_ROOT}'
sys.path.insert(0, str(REPO_ROOT))
os.chdir(REPO_ROOT)
print('cwd =', os.getcwd())

Mounted at /content/drive
cwd = /content/drive/MyDrive/caltech/junior/hw3


In [3]:
import torch, torchvision

print('torch:', torch.__version__, '| cuda:', torch.cuda.is_available())
torchvision.ops.nms(torch.zeros(0, 4), torch.zeros(0), 0.5)

torch: 2.10.0+cu128 | cuda: True


tensor([], dtype=torch.int64)

## 3. Configuration

Trains `rope2d`, then reads `learned` and `rope` history files from Drive (if present) for the 3-way table.

In [4]:
import yaml

CONFIG_PATH = REPO_ROOT / 'configs' / 'clip_eurosat.yaml'
assert CONFIG_PATH.exists()
with open(CONFIG_PATH) as f:
    cfg = yaml.safe_load(f)

TRAIN_IMG_SIZE = cfg['vit']['img_size']    # 64
EVAL_IMG_SIZE = 96
PATCH_SIZE = cfg['vit']['patch_size']      # 8
PE_TO_TRAIN = ['rope2d']                    # learned and rope are reused if available
PE_TO_COMPARE = ['learned', 'rope', 'rope2d']

print(f'train img_size = {TRAIN_IMG_SIZE} ({TRAIN_IMG_SIZE // PATCH_SIZE}x{TRAIN_IMG_SIZE // PATCH_SIZE} grid)')
print(f'eval  img_size = {EVAL_IMG_SIZE}  ({EVAL_IMG_SIZE  // PATCH_SIZE}x{EVAL_IMG_SIZE  // PATCH_SIZE} grid)')
print(f'training: {PE_TO_TRAIN}   comparing: {PE_TO_COMPARE}')

train img_size = 64 (8x8 grid)
eval  img_size = 96  (12x12 grid)
training: ['rope2d']   comparing: ['learned', 'rope', 'rope2d']


## 4. Train `rope2d` (and `rope` / `learned` if missing on Drive)

Set `OVERWRITE = True` to retrain from scratch. By default, each PE method is trained only if its `history.pt` isn't already on Drive.

In [5]:
import argparse, shutil, time

from scripts.pretrain_clip import train

OVERWRITE = False
device = 'cuda' if torch.cuda.is_available() else 'cpu'

pe_results = {}
for pe in PE_TO_COMPARE:
    local_dir = Path(f'/content/runs/clip_eurosat_{pe}')
    drive_dir = REPO_ROOT / 'runs' / local_dir.name
    history_path = drive_dir / 'history.pt'

    if not OVERWRITE and history_path.exists():
        hist = torch.load(history_path, map_location='cpu', weights_only=False)
        pe_results[pe] = hist
        extra = hist.get('extrapolation', {})
        print(f'[{pe}] reusing cached history  '
              f'(test_acc={hist["test_acc"]:.4f}, '
              f'extra@{EVAL_IMG_SIZE}={extra.get("test_acc")})')
        continue

    local_dir.mkdir(parents=True, exist_ok=True)
    args = argparse.Namespace(
        config=CONFIG_PATH,
        output_dir=local_dir,
        device=device,
        wandb=False,
        wandb_project='cs148b-hw3-rope2d',
        wandb_run_name=f'eurosat-{pe}',
        pos_encoding=pe,
        eval_img_size=EVAL_IMG_SIZE,
    )
    print(f'\n========== pos_encoding={pe} ==========')
    t0 = time.time()
    history = train(cfg, args)
    print(f'[{pe}] wall = {(time.time() - t0)/60:.1f} min  '
          f'test_acc={history["test_acc"]:.4f}  '
          f'extra@{EVAL_IMG_SIZE}={history["extrapolation"]["test_acc"]:.4f}')
    pe_results[pe] = history

    drive_dir.mkdir(parents=True, exist_ok=True)
    for f in local_dir.glob('*'):
        shutil.copy2(f, drive_dir / f.name)
    print(f'synced {local_dir} -> {drive_dir}')

[learned] reusing cached history  (test_acc=0.8986, extra@96=0.792997542997543)
[rope] reusing cached history  (test_acc=0.9060, extra@96=0.7764127764127764)

========== pos_encoding=rope2d ==========


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/105M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/34.8M [00:00<?, ?B/s]

...

epoch 20/20:   0%|          | 0/50 [00:00<?, ?it/s]

Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7f2ae6a54680>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
    if w.is_alive():
       ^^^^^^^^^^^^
  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
    assert self._parent_pid == os.getpid(), 'can only test a child process'
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
AssertionError: can only test a child process
Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7f2ae6a54680>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 16

[epoch 20/20] train_loss=3.3417  val_acc=0.9078  lr=0.00e+00  (15.6s)


Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7f2ae6a54680>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
    if w.is_alive():
       ^^^^^^^Exception ignored in: ^<function _MultiProcessingDataLoaderIter.__del__ at 0x7f2ae6a54680>^
^Traceback (most recent call last):
^  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
^
      File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
self._shutdown_workers()    
assert self._parent_pid == os.getpid(), 'can only test a child process'  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers

      if w.is_alive():
                ^^^^^^^^^^^^^^^^^^^^^^^^


[final] best_val_acc=0.9078  test_acc=0.9097
[extrapolation] running zero-shot eval at img_size=96
[extrapolation] img_size=96  val_acc=0.7723  test_acc=0.7721
[rope2d] wall = 6.9 min  test_acc=0.9097  extra@96=0.7721
synced /content/runs/clip_eurosat_rope2d -> /content/drive/MyDrive/caltech/junior/hw3/runs/clip_eurosat_rope2d


## 5. 3-row comparison table

In [6]:
cols = [
    ('pos_encoding',         12),
    (f'val@{TRAIN_IMG_SIZE}',  10),
    (f'test@{TRAIN_IMG_SIZE}', 11),
    (f'val@{EVAL_IMG_SIZE}',   10),
    (f'test@{EVAL_IMG_SIZE}',  11),
    ('delta_test',          11),
]
header = ' | '.join(f'{n:>{w}}' for n, w in cols)
print()
print(header)
print('-' * len(header))
for pe in PE_TO_COMPARE:
    h = pe_results.get(pe)
    if h is None:
        print(f'  (no history for {pe})')
        continue
    train_val = h['best_val_acc']
    train_test = h['test_acc']
    extra = h.get('extrapolation', {})
    eval_val = extra.get('val_acc', float('nan'))
    eval_test = extra.get('test_acc', float('nan'))
    delta = eval_test - train_test
    print(' | '.join([
        f'{pe:>12}',
        f'{train_val:>10.4f}',
        f'{train_test:>11.4f}',
        f'{eval_val:>10.4f}',
        f'{eval_test:>11.4f}',
        f'{delta:>+11.4f}',
    ]))


pos_encoding |     val@64 |     test@64 |     val@96 |     test@96 |  delta_test
--------------------------------------------------------------------------------
     learned |     0.9189 |      0.8986 |     0.7995 |      0.7930 |     -0.1057
        rope |     0.9041 |      0.9060 |     0.7649 |      0.7764 |     -0.1296
      rope2d |     0.9078 |      0.9097 |     0.7723 |      0.7721 |     -0.1376


## 6. Writeup notes (2–3 sentences)

Fill in using your actual numbers. Things to cover:

- **In-distribution (64×64):** Compare `rope2d` vs `rope` (1D) at the training size. The expected pattern is that 2D RoPE *matches or modestly beats* 1D RoPE — both encode positions via relative offsets, but 2D RoPE bakes in the right geometric prior (rotations along x and y separately), so attention dot products factor into separate horizontal- and vertical-offset components rather than being forced through a 1D row-major scan. On EuroSAT (where category-defining features like field boundaries / road geometry are spatially structured), this small inductive-bias advantage usually shows up as a few-tenths-of-a-point gain.

- **Length-extrapolation (96×96):** `rope2d` should extrapolate roughly as well as `rope` (1D) — both stay in-distribution for the *relative offsets* they were trained on, since the new grid is just "more of the same" offsets in each axis. Compare the `Δtest` column: both rope variants should beat learned-PE-with-interpolation by a wide margin. If 2D RoPE's `Δtest` is smaller (less negative) than 1D RoPE's, the explanation is that the 2D inductive bias degrades more gracefully when the grid stretches because each axis still encodes patch separation correctly, whereas 1D RoPE's row-major scan changes its row-stride (8 → 12) and the model never trained on stride-12 positional offsets.

Optional: note that the cost of 2D RoPE over 1D RoPE is negligible (same number of cos/sin lookups, just split half-and-half across two axes), so even small accuracy gains come essentially for free.